# Build Sequential Models

## Setup

In [ ]:
import torch


if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

device

In [ ]:
import gc

def del_vars(variable_names=[]):
    for name in variable_names:
        try:
            del globals()[name]
        except KeyError:
            pass  # ignore variables that have already been deleted
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()

## Prepare our data

In [ ]:
import os
import kagglehub
import pandas as pd
import datasets
from transformers import AutoTokenizer
import torch
from torch.utils.data import DataLoader

In [ ]:
data_path = kagglehub.dataset_download("julian3833/jigsaw-toxic-comment-classification-challenge",
                                        "train.csv",
                                        output_dir="../data"
                                        )

In [ ]:
dataset = datasets.load_dataset("csv", data_files=data_path)
dataset

In [ ]:
import re

# existing patterns
username_pattern = re.compile(r'\[\[user.*?\]\]|\[\[user talk.*?\]\]|user:\w+', re.IGNORECASE)
whitespace_pattern = re.compile(r'\s+')
url_pattern = re.compile(r'http\S+|www\.\S+')
repeat_char_pattern = re.compile(r'(.)\1{2,}')

# wiki markup patterns
wiki_link_pattern = re.compile(r'\[\[(?:[^\]|]*\|)?([^\]]+)\]\]')
wiki_template_pattern = re.compile(r'\{\{.*?\}\}')
wiki_bold_italic_pattern = re.compile(r"'{2,5}")
wiki_heading_pattern = re.compile(r'=+\s*(.*?)\s*=+')

def preprocess_batch(batch):
    cleaned_texts = []

    for text in batch["comment_text"]:
        text = text.lower()

        # remove usernames
        text = username_pattern.sub(" ", text)

        # replace URLs with a [URL] token
        text = url_pattern.sub("[URL]", text)

        # convert [[link|text]] -> text
        text = wiki_link_pattern.sub(r"\1", text)

        # remove templates {{...}}
        text = wiki_template_pattern.sub(" ", text)

        # remove bold/italic markup
        text = wiki_bold_italic_pattern.sub("", text)

        # remove headings
        text = wiki_heading_pattern.sub(r"\1", text)

        # normalize stretched characters
        text = repeat_char_pattern.sub(r"\1\1", text)

        # normalize whitespace
        text = whitespace_pattern.sub(" ", text).strip()

        cleaned_texts.append(text)

    batch["comment_text"] = cleaned_texts
    return batch

In [ ]:
dataset = dataset["train"].map(preprocess_batch, batched=True, num_proc=4)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
tokenizer.add_special_tokens({"additional_special_tokens": ["[URL]"]})

def tokenize_batch(batch):
    return tokenizer(
        batch["comment_text"],
        truncation=True,
        max_length=256
    )


In [ ]:
dataset = dataset.map(
    tokenize_batch,
    batched=True,
    num_proc=4
)
dataset = dataset.remove_columns(["comment_text", "id"])

dataset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "toxic",
        "severe_toxic",
        "obscene",
        "threat",
        "insult",
        "identity_hate"
    ]
)

In [ ]:
LABEL_COLUMNS = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]

def collate_fn(batch):
    input_ids = torch.stack([item["input_ids"] for item in batch])
    attention_mask = torch.stack([item["attention_mask"] for item in batch])

    labels = torch.stack([
        torch.tensor([item[label] for label in LABEL_COLUMNS], dtype=torch.float32)
        for item in batch
    ])

    X = {
        "input_ids": input_ids,
        "attention_mask": attention_mask
    }

    return X, labels

In [ ]:
split = dataset.train_test_split(test_size=0.2, seed=42)
train_set = split["train"]
test_valid_set = split["test"]
split = test_valid_set.train_test_split(test_size=0.5, seed=42)
valid_set = split["train"]
test_set = split["test"]

In [ ]:
print(f"Train Shape: {train_set.shape}")
print(f"Valid Shape: {valid_set.shape}")
print(f"Test Shape: {test_set.shape}")


In [ ]:
train_loader = DataLoader(train_set, batch_size=128, shuffle=True, collate_fn=collate_fn)
valid_loader = DataLoader(valid_set, batch_size=128, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_set, batch_size=128, shuffle=False, collate_fn=collate_fn)